<a href="https://colab.research.google.com/github/w39011651/LC-PLM/blob/colabLoRA/highparameter/LoRAFineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git lfs --exclude=* clone https://github.com/w39011651/LC-PLM

          with new flags from 'git clone'

'git clone' has been updated in upstream Git to have comparable
speeds to 'git lfs clone'.
fatal: destination path 'LC-PLM' already exists and is not an empty directory.
Error(s) during clone:
git clone failed: exit status 128


In [2]:
%cd LC-PLM
!git lfs pull

/content/LC-PLM


In [3]:
!git fetch origin
!git checkout LoRA
%cd ..

Already on 'LoRA'
Your branch is up to date with 'origin/LoRA'.
/content


In [4]:
!ls -l model.safetensors

ls: cannot access 'model.safetensors': No such file or directory


In [5]:
!pip install --no-build-isolation mamba-ssm

KeyboardInterrupt: 

In [ ]:
import torch
from transformers.models.auto.tokenization_auto import AutoTokenizer
from transformers.models.auto.modeling_auto import AutoModelForMaskedLM
from transformers.utils.quantization_config import BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn as nn
import typing

# Dataset Preparation

## Dataset Definition

In [ ]:
MAX_LEN = 128
PAD_TOKEN_ID = -100

In [ ]:
from torch.utils.data import Dataset, random_split

class ATPBingindDataset(Dataset):
    def __init__(self, tokenizer, sequences, labels, max_length = 128):
        #vocab_size default is 128 in model config and tokenizer, but 512 has the lowest eval loss in reference paper
        self.tokenizer = tokenizer
        self.sequences = sequences
        self.labels = labels
        self.max_length = max_length

    def __len__(self)->int:
        return len(self.sequences)

    def __getitem__(self, index):
        try:
            sequence = self.sequences[index]
            label = self.labels[index]

            encoding = self.tokenizer(
                sequence,
                padding='max_length',
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            )

            padded_label = torch.full((self.max_length, ), PAD_TOKEN_ID, dtype = torch.long)
            valid_length = min(len(label), self.max_length)
            padded_label[:valid_length] = torch.tensor([int(l) for l in label[:valid_length]])

            return {
                'input_ids': encoding['input_ids'].squeeze(0),
                'attention_mask': encoding['attention_mask'].squeeze(0),
                'labels': padded_label.clone().detach()
            }

        except Exception as e:
            print(f"Error at index {index}")
            raise e

## Split dataset function

In [ ]:
def split_dataset(dataset, split_size = 0.8):
    train_size = int(split_size * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    return train_dataset, test_dataset

# Data preprocess

In [ ]:
import json
with open('LC-PLM/dataprocess/ATP_rmsim.json') as f:
    data = json.load(f)

def generate_label(protein, positions)->typing.List:
    n = protein['sequence']['length']
    labels = [0 for _ in range(n)]

    for bind_site in positions:
        for i in range(bind_site[0], bind_site[1]+1):
            labels[i-1] = 1#0-index或者1-index?
    return labels

def get_binding_site(features)->typing.List:
    binding_site = []
    for feature in features:
        if feature['type'] == 'Binding site' and feature['ligand']['name']=='ATP':#only need FAD
            binding_site.append([feature['location']['start']['value'], feature['location']['end']['value']])
    return binding_site

## Generate label and split sequence
until the length of each subsequence <= MAX_LEN

In [ ]:
protein_information = []
ones_cnt = 0
zeros_cnt = 0
for protein in data['results']:
    info = {"sequence": [], "label": []}
    info['sequence'] = protein['sequence']['value']
    binding_site_interval = get_binding_site(protein['features'])
    info['label'] = generate_label(protein, binding_site_interval)

    if info['label'].count(0) == len(info['label']):
        continue

    ones_cnt += info['label'].count(1)
    zeros_cnt += info['label'].count(0)

    protein_information.append(info)

In [ ]:
sequences = []
labels = []

for item in protein_information:
    while len(item['sequence']) > MAX_LEN:
        sequences.append(item['sequence'][:MAX_LEN])
        labels.append(item['label'][:MAX_LEN])
        item['sequence'] = item['sequence'][MAX_LEN:]
        item['label'] = item['label'][MAX_LEN:]

    sequences.append(item['sequence'])
    labels.append(item['label'])

len(sequences[-1])

In [ ]:
import pandas as pd

df = pd.DataFrame(columns=['sample input'], data = [ones_cnt, zeros_cnt])
df

# Load Model and using LoRA

In [ ]:
MODEL_ID = "./LC-PLM"

quantization_config = BitsAndBytesConfig(
    load_in_8bit = True,
    # bnb_4bit_quant_type =  "nf4",
    # bnb_4bit_use_double_quant =  True,
    # bnb_4bit_compute_dtype = torch.bfloat16
)


base_model = AutoModelForMaskedLM.from_pretrained(MODEL_ID,
                          trust_remote_code = True,
                          device_map = "auto",
                          torch_dtype=torch.float32
                          )

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
dataset = ATPBingindDataset(tokenizer, sequences, labels, max_length=MAX_LEN)


In [ ]:
train_dataset, validate_dataset = split_dataset(dataset, 0.8)
validate_dataset, test_dataset = split_dataset(validate_dataset, 0.5)

In [ ]:
print(f'{len(train_dataset)} {len(validate_dataset)} {len(test_dataset)}')

In [ ]:
print(base_model) ## Check the projection layer in base model

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=[
        "mixer.mamba_fwd.in_proj",
        "mixer.mamba_fwd.x_proj",
        "mixer.mamba_fwd.dt_proj",
        "mixer.mamba_fwd.out_proj",
        "mixer.mamba_rev.in_proj",
        "mixer.mamba_rev.x_proj",
        "mixer.mamba_rev.dt_proj",
        "mixer.mamba_rev.out_proj"
    ],
    bias="none"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("Loading model complete")
print(f"Device Type:{type(model)}")
print(f"Device:{next(model.parameters()).device}")

## Define Sequence Labeling Class

In [ ]:
class LCPLMLoRAForSequenceLabeling(nn.Module):
    def __init__(self, base_model, num_labels=2):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(base_model.config.d_model, num_labels)#後續可以替換成簡單的CNN分類
        self.drop_out = nn.Dropout(0.1)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask = None, labels = None):
        outputs = self.base_model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            output_hidden_states = True
        )
        # Get the output from basemodel
        # This part is with gradient or no gradient? It should be no gradient because the limit of GPU's memory
        sequence_output = outputs.hidden_states[-1]
        logits = self.classifier(sequence_output)
        loss = None
        if labels is not None:# 後續可使用Mask遮罩來加強訓練
            active_logits = logits.view(-1, logits.size(-1))
            active_labels = labels.view(-1)
            loss = self.loss_fn(active_logits, active_labels)

        return loss, logits

In [ ]:
sequence_model = LCPLMLoRAForSequenceLabeling(base_model, num_labels=2)

## Custom Evaluation function and find the best threshold

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, roc_curve, matthews_corrcoef
from sklearn.metrics import roc_curve

def find_best_evalution(logits, labels, mode = 'gmean'):
    logits_np = np.array(logits)
    labels_np = np.array(labels)
    probs = 1 / (1+ np.exp(-logits_np[:,:,1]))

    labels_flat = labels_np.flatten()
    probs_flat = probs.flatten()

    active_indices = labels_flat != -100 # ignore mask
    final_labels = labels_flat[active_indices]
    final_probs = probs_flat[active_indices]

    fpr, tpr, thresholds = roc_curve(final_labels, final_probs)
    if mode == 'gmean':
        gmeans = np.sqrt(tpr * (1-fpr))
        ix = np.argmax(gmeans)
        best_threshold = thresholds[ix]
        return best_threshold
    elif mode == 'mcc':
        mcc_score_at_each_threshold = []

        for threshold in thresholds:
            predicted_classes_at_threshold = (final_probs > threshold).astype(int)
            mcc = matthews_corrcoef(final_labels, predicted_classes_at_threshold)
            mcc_score_at_each_threshold.append(mcc)

        max_mcc_score_index = np.argmax(mcc_score_at_each_threshold)
        best_threshold = thresholds[max_mcc_score_index]
        best_mcc = mcc_score_at_each_threshold[max_mcc_score_index]

        return best_threshold
    else:
        return 0.5

In [ ]:
def custom_classification_report(logits, labels, mode):
    labels_np = np.array(labels)
    logits_np = np.array(logits)

    threshold = find_best_evalution(logits, labels, mode)

    probs = 1/(1+np.exp(-logits_np[:,:,1]))
    preds = (probs > threshold).astype(int).flatten()
    labels_flat = labels_np.flatten()


    active_indices = labels_flat != -100
    final_labels = labels_flat[active_indices]
    final_preds = preds[active_indices]
    report_str = classification_report(final_labels, final_preds)
    report_dict = classification_report(final_labels, final_preds, output_dict=True)
    print(report_str)
    return report_dict

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            inputs = {k:v.to(device) for k, v in batch.items() if k != "labels"}
            labels = batch['labels'].to(device)

            loss, _ = model(**inputs, labels=labels)
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)

    return total_loss / total_samples

In [ ]:
from transformers.training_args import TrainingArguments
from transformers.trainer import Trainer
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim
from transformers.optimization import get_linear_schedule_with_warmup

def train(model):
    # Training Arguments 訓練參數
    output_dir="./results"
    num_train_epochs=3
    per_device_train_batch_size=4
    per_device_eval_batch_size=8
    weight_decay=0.01
    logging_dir="./logs"
    logging_steps=80
    eval_steps=500
    save_steps=1000
    eval_strategy="steps"
    save_strategy="steps"
    load_best_model_at_end=True
    metric_for_best_model="eval_loss"
    greater_is_better=False
    learning_rate=5e-4
    warmup_steps=100
    gradient_accumulation_steps=2
    fp16=True  # 使用混合精度訓練, Mamba seems like doesn't support mix precision,using fp32 will exceed limitation of VRAM
    remove_unused_columns=False  # 重要：保留自訂欄位
    dataloader_pin_memory=False
    report_to=None  # 關閉 wandb 等報告

    # Device setting 設備設定
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    # Create DataLoader 創建 DataLoader
    train_loader = DataLoader(train_dataset, per_device_train_batch_size, shuffle=True)
    validate_loader = DataLoader(validate_dataset, per_device_eval_batch_size, shuffle=True)
    total_steps = len(train_loader) * num_train_epochs

    # Calculate total steps
    total_steps = len(train_loader) * num_train_epochs

    # Setting Optimizer and Scheduler
    optimizer = optim.AdamW(
        model.parameters(),
        lr = learning_rate,
        weight_decay = weight_decay
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # Early Stopping parameter
    patience = 10
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state_dict = None
    global_steps = 0

    print(f"Starting Training...")
    print(f"Total Training steps: {total_steps}")
    print(f"Each Epoch steps: {len(train_loader)}")

    for epoch in range(num_train_epochs):
        sequence_model.train()
        total_train_loss = 0

        for steps, batch in enumerate(tqdm(train_loader, desc = f"Epoch {epoch+1}")):
            ##########################Training Part Below##########################
            inputs = {k:v.to(device) for k, v in batch.items() if k != "labels"}
            labels = batch['labels'].to(device)

            loss, _ = sequence_model(**inputs, labels=labels)
            loss = loss / gradient_accumulation_steps
            loss.backward()
            total_train_loss += loss.item()
            global_steps += 1

            if global_steps % logging_steps == 0:
                avg_train_loss = total_train_loss / (steps + 1)
                current_lr = scheduler.get_last_lr()[0]
                print(f"Epoch {epoch+1}, Step {steps + 1}, "
                      f"Training Loss: {avg_train_loss:.6f}, "
                      f"LR: {current_lr:.2e}")

            if (steps + 1) % gradient_accumulation_steps == 0:
                nn.utils.clip_grad_norm_(sequence_model.parameters(), max_norm=1.0)# 梯度裁減
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            ##########################Training Part Above##########################
            ##########################Validate Part Below##########################
            if (global_steps % eval_steps == 0):
                val_loss = evaluate(sequence_model, validate_loader, device)
                print(f"Validate loss: {val_loss:.6f}")

                # Early Stopping Checking
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    epochs_no_improve = 0
                    best_model_state_dict = sequence_model.state_dict().copy()
                    torch.save(best_model_state_dict, "atp_binding_model_lora_mamba.pt")
                else:
                    epochs_no_improve += 1

                if epochs_no_improve >= patience:
                    print("No improvement found during training.")
                    break
                sequence_model.train()
            ##########################Validate Part Above##########################
    if best_model_state_dict is not None:
        sequence_model.load_state_dict(best_model_state_dict)
        print(f'Loading best model, eval loss is: {best_val_loss}')

    return sequence_model




In [ ]:
trained_model = train(sequence_model)